# Task 1: GPT-Style Language Model From Scratch

Dataset: TinyStories

This project implements a character-level GPT-style language model from scratch.

Requirements:
- Character-level tokenization
- Custom character-to-index dictionaries
- Manual multi-head self-attention
- Causal masking
- Layer normalization
- Feed-forward network
- Residual connections
- Learnable token and positional embeddings
- Minimum 10 training epochs
- Training and validation evaluation
- Text generation and failure analysis

No prebuilt Transformer or attention modules will be used.

In [1]:

import os
import sys
import time
import random
import platform
import numpy as np
import torch

# Assignment identifiers
SID4 = 4349
SEED = SID4
SLICE = SID4 % 1000
HP_ID = 5
CLS_A = 9
CLS_B = 2

# Required training seeds
TRAIN_SEEDS = [4349, 4350, 4351]

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DATA266 Task 1 - GPT From Scratch")
print(f"SID4: {SID4}")
print(f"SEED: {SEED}")
print(f"SLICE: {SLICE}")
print(f"HP_ID: {HP_ID}")
print(f"CLS_A: {CLS_A}")
print(f"CLS_B: {CLS_B}")
print(f"Training seeds: {TRAIN_SEEDS}")
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Operating system: {platform.platform()}")
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU: Not available")

DATA266 Task 1 - GPT From Scratch
SID4: 4349
SEED: 4349
SLICE: 349
HP_ID: 5
CLS_A: 9
CLS_B: 2
Training seeds: [4349, 4350, 4351]
Python version: 3.13.3
PyTorch version: 2.14.0+cu132
NumPy version: 2.5.3
Operating system: Windows-11-10.0.26200-SP0
Device: cuda
GPU: NVIDIA GeForce RTX 4090


## Model Configuration

This model uses a deeper, compact architecture with a longer context window.

The main design choices are:
- 6 Transformer blocks
- 192-dimensional embeddings
- 3 attention heads
- 256-character context length
- 768-dimensional feed-forward layers
- 0.15 dropout
- Pre-LayerNorm for training stability

The model will be trained using AdamW with learning-rate warm-up and cosine decay.

In [2]:
CONFIG = {
    # Dataset
    "train_sequences": 100_000,
    "val_sequences": 10_000,
    "block_size": 256,

    # GPT architecture
    "n_layers": 6,
    "d_model": 192,
    "n_heads": 3,
    "d_ff": 768,
    "dropout": 0.15,
    "activation": "gelu",
    "normalization": "pre_layer_norm",

    # Training
    "batch_size": 64,
    "epochs": 10,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "warmup_steps": 1000,
    "gradient_clip": 1.0,

    # Text generation
    "temperature": 0.8,
    "generation_length": 300,

    # Reproducibility
    "seed": SEED,
}

assert CONFIG["d_model"] % CONFIG["n_heads"] == 0

print("Task 1 configuration:")

for key, value in CONFIG.items():
    print(f"{key}: {value}")

head_dim = CONFIG["d_model"] // CONFIG["n_heads"]
print(f"Attention head dimension: {head_dim}")

Task 1 configuration:
train_sequences: 100000
val_sequences: 10000
block_size: 256
n_layers: 6
d_model: 192
n_heads: 3
d_ff: 768
dropout: 0.15
activation: gelu
normalization: pre_layer_norm
batch_size: 64
epochs: 10
learning_rate: 0.0003
weight_decay: 0.01
warmup_steps: 1000
gradient_clip: 1.0
temperature: 0.8
generation_length: 300
seed: 4349
Attention head dimension: 64


## Load and Inspect TinyStories

TinyStories will be loaded in streaming mode so we can inspect examples without loading the entire dataset into memory.

In [5]:
import sys
!{sys.executable} -m pip install -q datasets

from datasets import load_dataset
from itertools import islice

tiny_stories_stream = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
)

print("TinyStories dataset loaded successfully.")
print(tiny_stories_stream)

sample_examples = list(islice(tiny_stories_stream, 3))

for i, example in enumerate(sample_examples):
    print(f"\nExample {i + 1}:")
    print(example["text"][:500])


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\yashashree2\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\yashashree2\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yashashree2\.cache\huggingface\hub\datasets--roneneldan--TinyStories. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/

TinyStories dataset loaded successfully.
IterableDataset({
    features: ['text'],
    num_shards: 4
})

Example 1:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b

Example 2:
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were falling. Beep liked how the leaves fall and wanted to play with them. Beep drove un

## Create Individual Train and Validation Sequences

Each member must create their own split. We shuffle TinyStories using the assignment seed and then collect fixed-length character sequences.

For each sequence:

- Input: characters 0 through 255
- Target: characters 1 through 256

In [6]:
from datasets import load_dataset

TRAIN_SEQUENCE_COUNT = CONFIG["train_sequences"]
VAL_SEQUENCE_COUNT = CONFIG["val_sequences"]
BLOCK_SIZE = CONFIG["block_size"]

def collect_fixed_length_sequences(
    dataset_stream,
    train_count,
    val_count,
    block_size
):
    total_needed = train_count + val_count
    collected_sequences = []

    for example in dataset_stream:
        text = example["text"]

        # Normalize line endings and remove unnecessary outer whitespace
        text = text.replace("\r\n", "\n").strip()

        if len(text) <= block_size:
            continue

        # Use non-overlapping chunks from each story
        for start in range(0, len(text) - block_size, block_size):
            sequence = text[start:start + block_size + 1]

            if len(sequence) == block_size + 1:
                collected_sequences.append(sequence)

            if len(collected_sequences) >= total_needed:
                train_sequences = collected_sequences[:train_count]
                val_sequences = collected_sequences[train_count:]
                return train_sequences, val_sequences

    raise RuntimeError(
        f"Only collected {len(collected_sequences)} sequences, "
        f"but needed {total_needed}."
    )


# Reload and shuffle the dataset using your assignment seed
shuffled_tiny_stories = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
).shuffle(
    seed=SEED,
    buffer_size=10_000
)

train_text_sequences, val_text_sequences = collect_fixed_length_sequences(
    dataset_stream=shuffled_tiny_stories,
    train_count=TRAIN_SEQUENCE_COUNT,
    val_count=VAL_SEQUENCE_COUNT,
    block_size=BLOCK_SIZE
)

print("Training sequences:", len(train_text_sequences))
print("Validation sequences:", len(val_text_sequences))
print("Characters per sequence:", len(train_text_sequences[0]))
print("\nTraining example:")
print(repr(train_text_sequences[0][:100]))

print("\nValidation example:")
print(repr(val_text_sequences[0][:100]))

Training sequences: 100000
Validation sequences: 10000
Characters per sequence: 257

Training example:
'Ben and Lily were hungry. They saw a cake on the table. It looked yummy. They wanted to eat it.\n\n"Le'

Validation example:
'she saw the farmer trying to prevent the cows from getting the milk from the tall bucket. \n\nPolly wa'


## Character-Level Vocabulary and Encoding

Each unique character receives an integer ID.

Two dictionaries are created:

- `char_to_idx`: character to integer
- `idx_to_char`: integer to character

For each 257-character sequence:

- Input contains characters 0–255
- Target contains characters 1–256

In [7]:
# Build the vocabulary from your individual train and validation sequences
all_text = "".join(train_text_sequences + val_text_sequences)

vocabulary = sorted(set(all_text))

char_to_idx = {
    character: index
    for index, character in enumerate(vocabulary)
}

idx_to_char = {
    index: character
    for index, character in enumerate(vocabulary)
}

VOCAB_SIZE = len(vocabulary)

print("Vocabulary size:", VOCAB_SIZE)
print("First vocabulary characters:", repr(vocabulary[:20]))
print("Character-to-index examples:")

for character in vocabulary[:10]:
    print(repr(character), "->", char_to_idx[character])

assert len(char_to_idx) == len(idx_to_char)
assert VOCAB_SIZE < 32_767

Vocabulary size: 101
First vocabulary characters: ['\t', '\n', ' ', '!', '"', '$', '&', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4']
Character-to-index examples:
'\t' -> 0
'\n' -> 1
' ' -> 2
'!' -> 3
'"' -> 4
'$' -> 5
'&' -> 6
"'" -> 7
'(' -> 8
')' -> 9


In [8]:
def encode_sequence_list(text_sequences, char_to_idx):
    input_sequences = []
    target_sequences = []

    for sequence in text_sequences:
        input_ids = [
            char_to_idx[character]
            for character in sequence[:-1]
        ]

        target_ids = [
            char_to_idx[character]
            for character in sequence[1:]
        ]

        input_sequences.append(input_ids)
        target_sequences.append(target_ids)

    # int16 saves memory; we will convert batches to int64 before embedding
    input_tensor = torch.tensor(input_sequences, dtype=torch.int16)
    target_tensor = torch.tensor(target_sequences, dtype=torch.int16)

    return input_tensor, target_tensor


train_inputs, train_targets = encode_sequence_list(
    train_text_sequences,
    char_to_idx
)

val_inputs, val_targets = encode_sequence_list(
    val_text_sequences,
    char_to_idx
)

print("Train input shape:", tuple(train_inputs.shape))
print("Train target shape:", tuple(train_targets.shape))
print("Validation input shape:", tuple(val_inputs.shape))
print("Validation target shape:", tuple(val_targets.shape))
print("Input tensor dtype:", train_inputs.dtype)

Train input shape: (100000, 256)
Train target shape: (100000, 256)
Validation input shape: (10000, 256)
Validation target shape: (10000, 256)
Input tensor dtype: torch.int16


## Create PyTorch Datasets and DataLoaders

The DataLoader divides the sequences into batches.

Each batch will contain:

- 64 input sequences
- 64 target sequences
- 256 character positions per sequence

The tensors are converted from `int16` to `int64` before entering the embedding layer.

In [9]:
from torch.utils.data import Dataset, DataLoader


class CharacterSequenceDataset(Dataset):
    def __init__(self, input_tensor, target_tensor):
        self.input_tensor = input_tensor
        self.target_tensor = target_tensor

    def __len__(self):
        return self.input_tensor.shape[0]

    def __getitem__(self, index):
        input_ids = self.input_tensor[index].long()
        target_ids = self.target_tensor[index].long()

        return input_ids, target_ids


train_dataset = CharacterSequenceDataset(
    train_inputs,
    train_targets
)

val_dataset = CharacterSequenceDataset(
    val_inputs,
    val_targets
)

data_generator = torch.Generator()
data_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    generator=data_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 1563
Validation batches: 157


In [12]:
sample_inputs, sample_targets = next(iter(train_loader))

sample_inputs = sample_inputs.to(
    DEVICE,
    non_blocking=True
)

sample_targets = sample_targets.to(
    DEVICE,
    non_blocking=True
)

print("Input batch shape:", tuple(sample_inputs.shape))
print("Target batch shape:", tuple(sample_targets.shape))
print("Input batch dtype:", sample_inputs.dtype)
print("Target batch dtype:", sample_targets.dtype)
print("Input batch device:", sample_inputs.device)
print("Target batch device:", sample_targets.device)

Input batch shape: (64, 256)
Target batch shape: (64, 256)
Input batch dtype: torch.int64
Target batch dtype: torch.int64
Input batch device: cuda:0
Target batch device: cuda:0


## Manual Causal Multi-Head Self-Attention

This layer manually implements:

- Query, key, and value projections
- Multi-head attention
- Scaled dot-product attention
- Causal masking
- Attention dropout
- Output projection

No prebuilt attention or Transformer modules are used.

In [13]:
import math
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        block_size,
        dropout
    ):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        # Creates query, key, and value together
        self.qkv_projection = nn.Linear(
            d_model,
            3 * d_model
        )

        self.output_projection = nn.Linear(
            d_model,
            d_model
        )

        self.attention_dropout = nn.Dropout(dropout)
        self.output_dropout = nn.Dropout(dropout)

        # Lower-triangular causal mask
        causal_mask = torch.tril(
            torch.ones(
                block_size,
                block_size,
                dtype=torch.bool
            )
        )

        self.register_buffer(
            "causal_mask",
            causal_mask,
            persistent=False
        )

    def forward(self, x, return_attention=False):
        batch_size, sequence_length, _ = x.shape

        qkv = self.qkv_projection(x)

        # Shape:
        # (batch_size, sequence_length, 3, n_heads, head_dim)
        qkv = qkv.view(
            batch_size,
            sequence_length,
            3,
            self.n_heads,
            self.head_dim
        )

        # Shape:
        # (3, batch_size, n_heads, sequence_length, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        query, key, value = qkv.unbind(dim=0)

        # Scaled dot-product attention
        attention_scores = (
            query @ key.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        # Apply causal mask
        current_mask = self.causal_mask[
            :sequence_length,
            :sequence_length
        ]

        attention_scores = attention_scores.masked_fill(
            ~current_mask,
            torch.finfo(attention_scores.dtype).min
        )

        attention_weights = F.softmax(
            attention_scores,
            dim=-1
        )

        attention_weights = self.attention_dropout(
            attention_weights
        )

        attention_output = attention_weights @ value

        # Combine attention heads
        attention_output = attention_output.transpose(
            1,
            2
        ).contiguous()

        attention_output = attention_output.view(
            batch_size,
            sequence_length,
            self.d_model
        )

        attention_output = self.output_projection(
            attention_output
        )

        attention_output = self.output_dropout(
            attention_output
        )

        if return_attention:
            return attention_output, attention_weights

        return attention_output

In [14]:
attention_test = CausalSelfAttention(
    d_model=CONFIG["d_model"],
    n_heads=CONFIG["n_heads"],
    block_size=CONFIG["block_size"],
    dropout=CONFIG["dropout"]
).to(DEVICE)

test_input = torch.randn(
    2,
    CONFIG["block_size"],
    CONFIG["d_model"],
    device=DEVICE
)

test_output, test_weights = attention_test(
    test_input,
    return_attention=True
)

print("Input shape:", tuple(test_input.shape))
print("Output shape:", tuple(test_output.shape))
print("Attention weights shape:", tuple(test_weights.shape))
print("Device:", test_output.device)

assert test_output.shape == test_input.shape

assert test_weights.shape == (
    2,
    CONFIG["n_heads"],
    CONFIG["block_size"],
    CONFIG["block_size"]
)

Input shape: (2, 256, 192)
Output shape: (2, 256, 192)
Attention weights shape: (2, 3, 256, 256)
Device: cuda:0


In [15]:
attention_test.eval()

with torch.no_grad():
    _, causal_weights = attention_test(
        test_input,
        return_attention=True
    )

# Select all positions above the main diagonal.
# These represent attention to future tokens.
future_attention_weights = torch.triu(
    causal_weights,
    diagonal=1
)

maximum_future_attention = future_attention_weights.abs().max().item()

print(
    "Maximum attention weight assigned to future tokens:",
    maximum_future_attention
)

assert maximum_future_attention < 1e-7

Maximum attention weight assigned to future tokens: 0.0


## Feed-Forward Network

The feed-forward network independently transforms each character position.

It contains:

1. Linear layer from 192 to 768 dimensions
2. GELU activation
3. Linear layer from 768 back to 192 dimensions
4. Dropout

The same feed-forward network structure is used inside every Transformer block.

In [16]:
class FeedForwardNetwork(nn.Module):
    def __init__(
        self,
        d_model,
        d_ff,
        dropout
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)

## Transformer Block

Each Transformer block uses:

- Pre-LayerNorm
- Causal multi-head self-attention
- Residual connection
- Second LayerNorm
- Feed-forward network
- Second residual connection

The residual structure is:

x = x + Attention(LayerNorm(x))
x = x + FeedForward(LayerNorm(x))

In [17]:
class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        block_size,
        dropout
    ):
        super().__init__()

        self.layer_norm_attention = nn.LayerNorm(d_model)
        self.attention = CausalSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            block_size=block_size,
            dropout=dropout
        )

        self.layer_norm_feed_forward = nn.LayerNorm(d_model)
        self.feed_forward = FeedForwardNetwork(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

    def forward(self, x):
        # Pre-LayerNorm attention sublayer with residual connection
        x = x + self.attention(
            self.layer_norm_attention(x)
        )

        # Pre-LayerNorm feed-forward sublayer with residual connection
        x = x + self.feed_forward(
            self.layer_norm_feed_forward(x)
        )

        return x

In [18]:
transformer_block_test = TransformerBlock(
    d_model=CONFIG["d_model"],
    n_heads=CONFIG["n_heads"],
    d_ff=CONFIG["d_ff"],
    block_size=CONFIG["block_size"],
    dropout=CONFIG["dropout"]
).to(DEVICE)

block_test_input = torch.randn(
    2,
    CONFIG["block_size"],
    CONFIG["d_model"],
    device=DEVICE
)

block_test_output = transformer_block_test(
    block_test_input
)

print("Input shape:", tuple(block_test_input.shape))
print("Output shape:", tuple(block_test_output.shape))
print("Device:", block_test_output.device)

assert block_test_output.shape == block_test_input.shape

Input shape: (2, 256, 192)
Output shape: (2, 256, 192)
Device: cuda:0


## Complete GPT-Style Language Model

The complete model converts character IDs into logits for every possible character in the vocabulary.

For each position, the model predicts the next character.

Output shape:

```text
(batch_size, sequence_length, vocabulary_size)

In [20]:

class GPTLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        block_size,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.block_size = block_size

        # Learnable token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learnable positional embeddings
        self.position_embedding = nn.Embedding(
            block_size,
            d_model
        )

        self.embedding_dropout = nn.Dropout(dropout)

        # Stack of Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                block_size=block_size,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.final_layer_norm = nn.LayerNorm(d_model)

        # Projects hidden states to vocabulary logits
        self.language_model_head = nn.Linear(
            d_model,
            vocab_size
        )

        self.apply(self._initialize_weights)

    def _initialize_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self, input_ids, target_ids=None):
        batch_size, sequence_length = input_ids.shape

        assert sequence_length <= self.block_size

        positions = torch.arange(
            sequence_length,
            device=input_ids.device
        ).unsqueeze(0)

        token_embeddings = self.token_embedding(input_ids)
        position_embeddings = self.position_embedding(positions)

        x = token_embeddings + position_embeddings
        x = self.embedding_dropout(x)

        for block in self.transformer_blocks:
            x = block(x)

        x = self.final_layer_norm(x)

        logits = self.language_model_head(x)

        loss = None

        if target_ids is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, self.vocab_size),
                target_ids.reshape(-1)
            )

        return logits, loss

In [25]:
model = GPTLanguageModel(
    vocab_size=VOCAB_SIZE,
    d_model=CONFIG["d_model"],
    n_layers=CONFIG["n_layers"],
    n_heads=CONFIG["n_heads"],
    d_ff=CONFIG["d_ff"],
    block_size=CONFIG["block_size"],
    dropout=CONFIG["dropout"]
).to(DEVICE)

model_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Model device:", next(model.parameters()).device)
print("Vocabulary size:", VOCAB_SIZE)
print(
    "Total parameters:",
    f"{model_parameter_count:,}"
)

model.eval()

with torch.no_grad():
    model_test_logits, model_test_loss = model(
        sample_inputs,
        sample_targets
    )

print("Input shape:", tuple(sample_inputs.shape))
print("Logits shape:", tuple(model_test_logits.shape))
print("Initial loss:", model_test_loss.item())

Model device: cuda:0
Vocabulary size: 101
Total parameters: 2,757,605
Input shape: (64, 256)
Logits shape: (64, 256, 101)
Initial loss: 4.740846157073975


## Autoregressive Text Generation

The model generates text step by step.

At every step:

1. Use the current context as input.
2. Predict the next-character logits.
3. Apply temperature scaling.
4. Convert logits to probabilities.
5. Sample the next character.
6. Append it to the context.

The model is currently untrained, so this first generated sample will not be meaningful. It only tests whether generation works.

In [26]:
@torch.no_grad()
def generate_text(
    model,
    prompt,
    max_new_tokens,
    temperature,
    char_to_idx,
    idx_to_char,
    block_size,
    device
):
    model.eval()

    # Make sure every prompt character is in the vocabulary
    unknown_characters = [
        character
        for character in prompt
        if character not in char_to_idx
    ]

    if unknown_characters:
        raise ValueError(
            f"Prompt contains unknown characters: "
            f"{unknown_characters}"
        )

    input_ids = torch.tensor(
        [[char_to_idx[character] for character in prompt]],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):
        context_ids = input_ids[:, -block_size:]

        logits, _ = model(context_ids)

        # Use only the prediction for the final position
        next_token_logits = logits[:, -1, :]

        # Temperature scaling
        next_token_logits = next_token_logits / temperature

        probabilities = F.softmax(
            next_token_logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

    generated_characters = [
        idx_to_char[token_id]
        for token_id in input_ids[0].tolist()
    ]

    return "".join(generated_characters)

In [27]:
torch.manual_seed(SEED)

test_generated_text = generate_text(
    model=model,
    prompt="Once upon a time",
    max_new_tokens=50,
    temperature=CONFIG["temperature"],
    char_to_idx=char_to_idx,
    idx_to_char=idx_to_char,
    block_size=CONFIG["block_size"],
    device=DEVICE
)

print(test_generated_text)

Once upon a timedB³nfRR9N!8ZX3AÃ;ÃV3¬fs_7;	i"sk7YRâ&p»GnÂ()isYJ¡Q?


## Optimizer and Learning-Rate Schedule

The model will use:

- AdamW optimizer
- Initial learning rate of `3e-4`
- Weight decay of `0.01`
- 1,000 warm-up steps
- Cosine learning-rate decay after warm-up

Warm-up gradually increases the learning rate at the beginning of training. Cosine decay then gradually reduces it to improve training stability.

In [28]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

total_training_steps = (
    CONFIG["epochs"] * len(train_loader)
)

warmup_steps = min(
    CONFIG["warmup_steps"],
    total_training_steps // 10
)

print("Total training steps:", total_training_steps)
print("Warm-up steps:", warmup_steps)


def get_learning_rate(step):
    # Linear warm-up
    if step < warmup_steps:
        return CONFIG["learning_rate"] * (
            (step + 1) / warmup_steps
        )

    # Cosine decay
    decay_steps = total_training_steps - warmup_steps
    current_decay_step = step - warmup_steps

    progress = current_decay_step / decay_steps
    progress = min(max(progress, 0.0), 1.0)

    cosine_factor = 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

    return CONFIG["learning_rate"] * cosine_factor


def update_optimizer_learning_rate(optimizer, learning_rate):
    for parameter_group in optimizer.param_groups:
        parameter_group["lr"] = learning_rate


print("Learning rate at step 0:", get_learning_rate(0))
print(
    "Learning rate at warm-up end:",
    get_learning_rate(warmup_steps)
)
print(
    "Learning rate at final step:",
    get_learning_rate(total_training_steps - 1)
)

Total training steps: 15630
Warm-up steps: 1000
Learning rate at step 0: 3e-07
Learning rate at warm-up end: 0.0003
Learning rate at final step: 3.458377167975257e-12


## Training and Validation Functions

The training function will:

- Move batches to the RTX 4090
- Apply the learning-rate schedule
- Calculate cross-entropy loss
- Perform backpropagation
- Clip gradients
- Track accuracy, gradient norms, loss spikes, NaNs, and speed

The validation function evaluates the model without updating its weights.

In [29]:
def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def train_one_epoch(
    model,
    data_loader,
    optimizer,
    global_step,
    max_batches=None
):
    model.train()

    total_loss = 0.0
    total_tokens = 0
    correct_predictions = 0
    gradient_norms = []
    nan_count = 0
    loss_spike_count = 0
    previous_loss = None

    start_time = time.perf_counter()

    for batch_index, (input_batch, target_batch) in enumerate(data_loader):

        if max_batches is not None and batch_index >= max_batches:
            break

        input_batch = input_batch.to(
            DEVICE,
            non_blocking=True
        )

        target_batch = target_batch.to(
            DEVICE,
            non_blocking=True
        )

        current_learning_rate = get_learning_rate(
            global_step
        )

        update_optimizer_learning_rate(
            optimizer,
            current_learning_rate
        )

        optimizer.zero_grad(set_to_none=True)

        logits, loss = model(
            input_batch,
            target_batch
        )

        if not torch.isfinite(loss):
            nan_count += 1
            global_step += 1
            continue

        loss_value = loss.item()

        if (
            previous_loss is not None
            and loss_value > previous_loss * 1.5
        ):
            loss_spike_count += 1

        previous_loss = loss_value

        loss.backward()

        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CONFIG["gradient_clip"]
        )

        gradient_norms.append(
            float(gradient_norm.item())
        )

        optimizer.step()

        batch_token_count = input_batch.numel()

        total_loss += loss_value * batch_token_count
        total_tokens += batch_token_count

        predicted_ids = logits.argmax(dim=-1)

        correct_predictions += (
            predicted_ids == target_batch
        ).sum().item()

        global_step += 1

    synchronize_device()

    elapsed_seconds = time.perf_counter() - start_time

    average_loss = total_loss / max(total_tokens, 1)
    accuracy = correct_predictions / max(total_tokens, 1)
    tokens_per_second = total_tokens / max(elapsed_seconds, 1e-9)

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "gradient_norm_mean": (
            sum(gradient_norms) / max(len(gradient_norms), 1)
        ),
        "gradient_norm_max": (
            max(gradient_norms)
            if gradient_norms
            else 0.0
        ),
        "nan_count": nan_count,
        "loss_spike_count": loss_spike_count,
        "tokens": total_tokens,
        "tokens_per_second": tokens_per_second,
        "elapsed_seconds": elapsed_seconds,
        "learning_rate": current_learning_rate,
        "global_step": global_step
    }


@torch.no_grad()
def evaluate_model(
    model,
    data_loader,
    max_batches=None
):
    model.eval()

    total_loss = 0.0
    total_tokens = 0
    correct_predictions = 0

    start_time = time.perf_counter()

    for batch_index, (input_batch, target_batch) in enumerate(data_loader):

        if max_batches is not None and batch_index >= max_batches:
            break

        input_batch = input_batch.to(
            DEVICE,
            non_blocking=True
        )

        target_batch = target_batch.to(
            DEVICE,
            non_blocking=True
        )

        logits, loss = model(
            input_batch,
            target_batch
        )

        batch_token_count = input_batch.numel()

        total_loss += loss.item() * batch_token_count
        total_tokens += batch_token_count

        predicted_ids = logits.argmax(dim=-1)

        correct_predictions += (
            predicted_ids == target_batch
        ).sum().item()

    synchronize_device()

    elapsed_seconds = time.perf_counter() - start_time

    average_loss = total_loss / max(total_tokens, 1)
    accuracy = correct_predictions / max(total_tokens, 1)

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "tokens": total_tokens,
        "elapsed_seconds": elapsed_seconds
    }

In [30]:
print("Running two-batch training smoke test...")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(DEVICE)

smoke_train_metrics = train_one_epoch(
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    global_step=0,
    max_batches=2
)

smoke_val_metrics = evaluate_model(
    model=model,
    data_loader=val_loader,
    max_batches=2
)

print("\nSmoke-test training results:")
for key, value in smoke_train_metrics.items():
    print(f"{key}: {value}")

print("\nSmoke-test validation results:")
for key, value in smoke_val_metrics.items():
    print(f"{key}: {value}")

if torch.cuda.is_available():
    peak_memory_gb = (
        torch.cuda.max_memory_allocated(DEVICE)
        / (1024 ** 3)
    )

    print(
        f"\nPeak GPU memory: {peak_memory_gb:.2f} GB"
    )

Running two-batch training smoke test...

Smoke-test training results:
loss: 4.7144880294799805
accuracy: 0.004150390625
gradient_norm_mean: 6.437023639678955
gradient_norm_max: 6.476375579833984
nan_count: 0
loss_spike_count: 0
tokens: 32768
tokens_per_second: 62352.956399360955
elapsed_seconds: 0.5255243999999948
learning_rate: 6e-07
global_step: 2

Smoke-test validation results:
loss: 4.732875108718872
accuracy: 0.002044677734375
tokens: 32768
elapsed_seconds: 0.021144500000445987

Peak GPU memory: 1.97 GB


In [31]:
# Reset random seeds
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Clear the smoke-test model
del model
del optimizer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Create a fresh model
model = GPTLanguageModel(
    vocab_size=VOCAB_SIZE,
    d_model=CONFIG["d_model"],
    n_layers=CONFIG["n_layers"],
    n_heads=CONFIG["n_heads"],
    d_ff=CONFIG["d_ff"],
    block_size=CONFIG["block_size"],
    dropout=CONFIG["dropout"]
).to(DEVICE)

# Create a fresh optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

global_step = 0

model_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Fresh model created.")
print("Model device:", next(model.parameters()).device)
print("Total parameters:", f"{model_parameter_count:,}")
print("Global step:", global_step)

Fresh model created.
Model device: cuda:0
Total parameters: 2,757,605
Global step: 0


## Official GPT Training

The model is trained for the required minimum of 10 epochs.

For every epoch, we record:

- Training loss
- Validation loss
- Training accuracy
- Validation accuracy
- Gradient norms
- NaN count
- Loss spikes
- Tokens per second
- Learning rate
- Training time

A checkpoint is saved after every epoch.

In [32]:
from pathlib import Path
import json

checkpoint_directory = Path("task1_checkpoints")
checkpoint_directory.mkdir(
    parents=True,
    exist_ok=True
)

history = []
global_step = 0
best_validation_loss = float("inf")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(DEVICE)

total_training_start = time.perf_counter()

for epoch in range(CONFIG["epochs"]):

    epoch_start = time.perf_counter()

    train_metrics = train_one_epoch(
        model=model,
        data_loader=train_loader,
        optimizer=optimizer,
        global_step=global_step
    )

    global_step = train_metrics["global_step"]

    validation_metrics = evaluate_model(
        model=model,
        data_loader=val_loader
    )

    epoch_elapsed_seconds = (
        time.perf_counter() - epoch_start
    )

    train_loss = train_metrics["loss"]
    validation_loss = validation_metrics["loss"]

    validation_perplexity = math.exp(
        min(validation_loss, 20)
    )

    epoch_record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "train_accuracy": train_metrics["accuracy"],
        "validation_accuracy": validation_metrics["accuracy"],
        "validation_perplexity": validation_perplexity,
        "gradient_norm_mean": train_metrics[
            "gradient_norm_mean"
        ],
        "gradient_norm_max": train_metrics[
            "gradient_norm_max"
        ],
        "nan_count": train_metrics["nan_count"],
        "loss_spike_count": train_metrics[
            "loss_spike_count"
        ],
        "tokens_per_second": train_metrics[
            "tokens_per_second"
        ],
        "learning_rate": train_metrics[
            "learning_rate"
        ],
        "epoch_elapsed_seconds": epoch_elapsed_seconds,
        "global_step": global_step
    }

    history.append(epoch_record)

    checkpoint = {
        "epoch": epoch + 1,
        "global_step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": CONFIG,
        "vocab_size": VOCAB_SIZE,
        "char_to_idx": char_to_idx,
        "idx_to_char": idx_to_char,
        "history": history
    }

    epoch_checkpoint_path = (
        checkpoint_directory
        / f"task1_epoch_{epoch + 1:02d}.pt"
    )

    torch.save(
        checkpoint,
        epoch_checkpoint_path
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss

        best_checkpoint_path = (
            checkpoint_directory
            / "task1_best_model.pt"
        )

        torch.save(
            checkpoint,
            best_checkpoint_path
        )

    print(
        f"Epoch {epoch + 1:02d}/{CONFIG['epochs']} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {validation_loss:.4f} | "
        f"Val perplexity: {validation_perplexity:.2f} | "
        f"Train accuracy: {train_metrics['accuracy']:.4f} | "
        f"Val accuracy: {validation_metrics['accuracy']:.4f} | "
        f"Tokens/sec: {train_metrics['tokens_per_second']:.0f} | "
        f"Time: {epoch_elapsed_seconds / 60:.2f} min"
    )

total_training_time = (
    time.perf_counter() - total_training_start
)

if torch.cuda.is_available():
    peak_memory_gb = (
        torch.cuda.max_memory_allocated(DEVICE)
        / (1024 ** 3)
    )
else:
    peak_memory_gb = 0.0

training_summary = {
    "total_training_time_seconds": total_training_time,
    "total_training_time_minutes": (
        total_training_time / 60
    ),
    "peak_gpu_memory_gb": peak_memory_gb,
    "total_epochs": CONFIG["epochs"],
    "total_steps": global_step,
    "parameter_count": model_parameter_count
}

with open(
    "task1_training_history.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            "history": history,
            "summary": training_summary
        },
        file,
        indent=2
    )

print("\nTraining complete.")
print(
    "Total training time:",
    f"{total_training_time / 60:.2f} minutes"
)
print(
    "Peak GPU memory:",
    f"{peak_memory_gb:.2f} GB"
)

Epoch 01/10 | Train loss: 2.0814 | Val loss: 1.1898 | Val perplexity: 3.29 | Train accuracy: 0.4208 | Val accuracy: 0.6316 | Tokens/sec: 607390 | Time: 0.73 min
Epoch 02/10 | Train loss: 1.1298 | Val loss: 0.9192 | Val perplexity: 2.51 | Train accuracy: 0.6503 | Val accuracy: 0.7127 | Tokens/sec: 606214 | Time: 0.73 min
Epoch 03/10 | Train loss: 0.9577 | Val loss: 0.8221 | Val perplexity: 2.28 | Train accuracy: 0.7015 | Val accuracy: 0.7415 | Tokens/sec: 605312 | Time: 0.73 min
Epoch 04/10 | Train loss: 0.8840 | Val loss: 0.7788 | Val perplexity: 2.18 | Train accuracy: 0.7234 | Val accuracy: 0.7548 | Tokens/sec: 602430 | Time: 0.73 min
Epoch 05/10 | Train loss: 0.8436 | Val loss: 0.7524 | Val perplexity: 2.12 | Train accuracy: 0.7354 | Val accuracy: 0.7625 | Tokens/sec: 603722 | Time: 0.73 min
Epoch 06/10 | Train loss: 0.8171 | Val loss: 0.7342 | Val perplexity: 2.08 | Train accuracy: 0.7431 | Val accuracy: 0.7679 | Tokens/sec: 609412 | Time: 0.72 min
Epoch 07/10 | Train loss: 0.7981 |

In [33]:
print("Running final evaluation on the full datasets...")

final_train_metrics = evaluate_model(
    model=model,
    data_loader=train_loader
)

final_validation_metrics = evaluate_model(
    model=model,
    data_loader=val_loader
)

final_train_loss = final_train_metrics["loss"]
final_validation_loss = final_validation_metrics["loss"]

final_train_perplexity = math.exp(
    min(final_train_loss, 20)
)

final_validation_perplexity = math.exp(
    min(final_validation_loss, 20)
)

final_train_bits_per_character = (
    final_train_loss / math.log(2)
)

final_validation_bits_per_character = (
    final_validation_loss / math.log(2)
)

generalization_gap = (
    final_validation_loss - final_train_loss
)

average_gradient_norm = sum(
    record["gradient_norm_mean"]
    for record in history
) / len(history)

maximum_gradient_norm = max(
    record["gradient_norm_max"]
    for record in history
)

total_nan_count = sum(
    record["nan_count"]
    for record in history
)

total_loss_spike_count = sum(
    record["loss_spike_count"]
    for record in history
)

final_metrics = {
    "training_cross_entropy_loss": final_train_loss,
    "validation_cross_entropy_loss": final_validation_loss,
    "training_perplexity": final_train_perplexity,
    "validation_perplexity": final_validation_perplexity,
    "training_bits_per_character": (
        final_train_bits_per_character
    ),
    "validation_bits_per_character": (
        final_validation_bits_per_character
    ),
    "generalization_gap": generalization_gap,
    "training_top_1_accuracy": (
        final_train_metrics["accuracy"]
    ),
    "validation_top_1_accuracy": (
        final_validation_metrics["accuracy"]
    ),
    "average_gradient_norm": average_gradient_norm,
    "maximum_gradient_norm": maximum_gradient_norm,
    "total_nan_count": total_nan_count,
    "total_loss_spike_count": total_loss_spike_count,
    "parameter_count": model_parameter_count,
    "training_tokens_per_second": (
        history[-1]["tokens_per_second"]
    ),
    "total_training_time_seconds": (
        total_training_time
    ),
    "peak_gpu_memory_gb": peak_memory_gb
}

for metric_name, metric_value in final_metrics.items():
    print(f"{metric_name}: {metric_value}")

Running final evaluation on the full datasets...
training_cross_entropy_loss: 0.7086054393005371
validation_cross_entropy_loss: 0.7076739616394043
training_perplexity: 2.031156711307936
validation_perplexity: 2.029265615098764
training_bits_per_character: 1.0223015532258304
validation_bits_per_character: 1.0209577150234153
generalization_gap: -0.0009314776611327735
training_top_1_accuracy: 0.7756405078125
validation_top_1_accuracy: 0.77588671875
average_gradient_norm: 0.6070396580798307
maximum_gradient_norm: 5.447399139404297
total_nan_count: 0
total_loss_spike_count: 0
parameter_count: 2757605
training_tokens_per_second: 610617.5098072446
total_training_time_seconds: 435.4786291999999
peak_gpu_memory_gb: 1.9706816673278809


## Text Generation and Diversity Metrics

Because this is a character-level model, diversity metrics are calculated over character n-grams.

- Distinct-1: unique characters / total characters
- Distinct-2: unique character bigrams / total bigrams
- Distinct-3: unique character trigrams / total trigrams
- Repeated 4-gram rate: proportion of repeated character 4-grams

In [34]:
def distinct_ngram_score(text, n):
    characters = list(text)

    if len(characters) < n:
        return 0.0

    ngrams = [
        tuple(characters[index:index + n])
        for index in range(len(characters) - n + 1)
    ]

    return len(set(ngrams)) / len(ngrams)


def repeated_ngram_rate(text, n=4):
    characters = list(text)

    if len(characters) < n:
        return 0.0

    ngrams = [
        tuple(characters[index:index + n])
        for index in range(len(characters) - n + 1)
    ]

    unique_count = len(set(ngrams))
    repeated_count = len(ngrams) - unique_count

    return repeated_count / len(ngrams)


generation_prompts = [
    "Once upon a time",
    "One day,",
    "The little"
]

generated_samples = []

model.eval()

for sample_index, prompt in enumerate(generation_prompts):

    torch.manual_seed(SEED + sample_index)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + sample_index)

    synchronize_device()
    generation_start = time.perf_counter()

    generated_text = generate_text(
        model=model,
        prompt=prompt,
        max_new_tokens=CONFIG["generation_length"],
        temperature=CONFIG["temperature"],
        char_to_idx=char_to_idx,
        idx_to_char=idx_to_char,
        block_size=CONFIG["block_size"],
        device=DEVICE
    )

    synchronize_device()

    generation_elapsed = (
        time.perf_counter() - generation_start
    )

    generated_token_count = CONFIG["generation_length"]

    generation_tokens_per_second = (
        generated_token_count
        / max(generation_elapsed, 1e-9)
    )

    sample_metrics = {
        "prompt": prompt,
        "text": generated_text,
        "distinct_1": distinct_ngram_score(
            generated_text,
            1
        ),
        "distinct_2": distinct_ngram_score(
            generated_text,
            2
        ),
        "distinct_3": distinct_ngram_score(
            generated_text,
            3
        ),
        "repeated_4gram_rate": repeated_ngram_rate(
            generated_text,
            4
        ),
        "generation_tokens_per_second": (
            generation_tokens_per_second
        )
    }

    generated_samples.append(sample_metrics)

    print(f"\nSample {sample_index + 1}")
    print("-" * 80)
    print(generated_text)
    print("\nGeneration metrics:")
    print(
        "Distinct-1:",
        sample_metrics["distinct_1"]
    )
    print(
        "Distinct-2:",
        sample_metrics["distinct_2"]
    )
    print(
        "Distinct-3:",
        sample_metrics["distinct_3"]
    )
    print(
        "Repeated 4-gram rate:",
        sample_metrics["repeated_4gram_rate"]
    )
    print(
        "Generation tokens/second:",
        sample_metrics[
            "generation_tokens_per_second"
        ]
    )


Sample 1
--------------------------------------------------------------------------------
Once upon a time, there was a little girl named Lily. She had a big smile on her house and a special wind. She loved to play in the woods, especially adventures. One day, she went by a race to the water. Every day, she saw a big tree with lots of money. The pink was a big pretty girl with a playground beautiful pla

Generation metrics:
Distinct-1: 0.0949367088607595
Distinct-2: 0.4793650793650794
Distinct-3: 0.7547770700636943
Repeated 4-gram rate: 0.14057507987220447
Generation tokens/second: 348.90475888658256

Sample 2
--------------------------------------------------------------------------------
One day, a little girl named Lily went to the park with her mom. They saw a big tree with a big rock. The tree was filled with a tall green ball on the ground. Lily was so happy that she saw that she wanted to go off to the ground.

As they walked, Lily saw her favourite tree and the little girl was

In [35]:
generation_metric_names = [
    "distinct_1",
    "distinct_2",
    "distinct_3",
    "repeated_4gram_rate",
    "generation_tokens_per_second"
]

average_generation_metrics = {}

for metric_name in generation_metric_names:
    metric_values = [
        sample[metric_name]
        for sample in generated_samples
    ]

    average_generation_metrics[metric_name] = (
        sum(metric_values) / len(metric_values)
    )

final_metrics.update(
    average_generation_metrics
)

print("Average generation metrics:")

for metric_name, metric_value in (
    average_generation_metrics.items()
):
    print(f"{metric_name}: {metric_value}")

# Save the exact generated samples
with open(
    "task1_generated_samples.txt",
    "w",
    encoding="utf-8"
) as file:

    for index, sample in enumerate(generated_samples, 1):
        file.write(f"Sample {index}\n")
        file.write("=" * 80)
        file.write("\n")
        file.write(f"Prompt: {sample['prompt']}\n\n")
        file.write(sample["text"])
        file.write("\n\n")
        file.write("Generation metrics:\n")

        for metric_name in generation_metric_names:
            file.write(
                f"{metric_name}: "
                f"{sample[metric_name]}\n"
            )

        file.write("\n\n")

print("\nSaved generated samples to:")
print("task1_generated_samples.txt")

Average generation metrics:
distinct_1: 0.09744643542111897
distinct_2: 0.4227535481828515
distinct_3: 0.6663279478940775
repeated_4gram_rate: 0.2309668663919612
generation_tokens_per_second: 351.2900545462971

Saved generated samples to:
task1_generated_samples.txt


## Sequence Model Failure Analysis

The generated text shows several failure patterns:

1. Broken grammar and semantic inconsistency
2. Loss of coherence and abrupt topic changes
3. Repetition and semantically incorrect descriptions

In [36]:
failure_analysis_text = """
# Task 1: Sequence Model Failure Analysis

## Failure Case 1: Broken Grammar and Semantic Inconsistency

Generated text snippet:

"She had a big smile on her house and a special wind."

Failure type:

Broken grammar and semantic inconsistency.

Observation:

The model produces a grammatically unusual and semantically incorrect
description. A smile is normally associated with a person, not a house,
and "a special wind" is not connected clearly to the sentence. This shows
that the model learned common words and sentence patterns but did not fully
learn how to maintain meaningful relationships between objects and actions.

## Failure Case 2: Loss of Coherence and Abrupt Topic Change

Generated text snippet:

"Every day, she saw a big tree with lots of money. The pink was a big
pretty girl with a playground beautiful pla"

Failure type:

Loss of coherence and incomplete generation.

Observation:

The text begins with a girl observing a tree, but the sentence suddenly
introduces money, a pink girl, and a playground without a logical transition.
The final phrase is also incomplete. This indicates that the model can
generate locally plausible phrases but struggles to maintain a consistent
storyline over a longer sequence.

## Failure Case 3: Repetition and Semantic Inconsistency

Generated text snippet:

"They saw a big tree with a big rock. The tree was filled with a tall green
ball on the ground. Lily was so happy that she saw that she wanted to go off
to the ground."

Failure type:

Repetition and semantic inconsistency.

Observation:

The model repeats words such as "big," "tree," and "ground." It also creates
an unusual description in which a tree is filled with a ball and produces an
unclear sentence about going "off to the ground." This suggests that the
model has learned frequent character and word patterns but does not always
preserve realistic object relationships or grammatical structure.

## Overall Observation

The model achieves low validation loss and approximately 77.6% next-character
accuracy, but the generated samples still contain grammar errors, repetition,
and coherence problems. This difference occurs because next-character accuracy
can be high even when the model makes errors that damage the meaning of a
longer story. Increasing the training data, model capacity, context quality,
or training duration could be tested as possible improvements.
"""

print(failure_analysis_text)

with open(
    "task1_failure_analysis.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(failure_analysis_text)

print("\nSaved failure analysis to task1_failure_analysis.md")


# Task 1: Sequence Model Failure Analysis

## Failure Case 1: Broken Grammar and Semantic Inconsistency

Generated text snippet:

"She had a big smile on her house and a special wind."

Failure type:

Broken grammar and semantic inconsistency.

Observation:

The model produces a grammatically unusual and semantically incorrect
description. A smile is normally associated with a person, not a house,
and "a special wind" is not connected clearly to the sentence. This shows
that the model learned common words and sentence patterns but did not fully
learn how to maintain meaningful relationships between objects and actions.

## Failure Case 2: Loss of Coherence and Abrupt Topic Change

Generated text snippet:

"Every day, she saw a big tree with lots of money. The pink was a big
pretty girl with a playground beautiful pla"

Failure type:

Loss of coherence and incomplete generation.

Observation:

The text begins with a girl observing a tree, but the sentence suddenly
introduces money, a p

## Save Metrics Report and Loss Curves

This cell saves:

- `metrics_report.csv`
- `task1_loss_curves.svg`
- `task1_model_configuration.json`

The SVG loss curve is created using pure Python.

In [37]:
import csv
import json
from pathlib import Path


# Save all final metrics to CSV
metrics_report_path = Path(
    "metrics_report.csv"
)

with open(
    metrics_report_path,
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.writer(file)
    writer.writerow(["metric", "value"])

    for metric_name, metric_value in final_metrics.items():
        writer.writerow([
            metric_name,
            metric_value
        ])


# Save model configuration
configuration_report = {
    "assignment_identifiers": {
        "SID4": SID4,
        "SEED": SEED,
        "SLICE": SLICE,
        "HP_ID": HP_ID,
        "CLS_A": CLS_A,
        "CLS_B": CLS_B
    },
    "model_configuration": CONFIG,
    "vocabulary_size": VOCAB_SIZE,
    "parameter_count": model_parameter_count,
    "device": str(DEVICE),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "Not available"
    )
}

with open(
    "task1_model_configuration.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        configuration_report,
        file,
        indent=2
    )


def create_loss_curve_svg(
    history,
    output_path,
    width=900,
    height=550
):
    margin_left = 80
    margin_right = 40
    margin_top = 50
    margin_bottom = 70

    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom

    train_losses = [
        record["train_loss"]
        for record in history
    ]

    validation_losses = [
        record["validation_loss"]
        for record in history
    ]

    all_losses = train_losses + validation_losses

    minimum_loss = min(all_losses)
    maximum_loss = max(all_losses)

    loss_range = maximum_loss - minimum_loss

    if loss_range == 0:
        loss_range = 1.0

    def point_coordinates(index, loss):
        if len(history) == 1:
            x_position = margin_left + plot_width / 2
        else:
            x_position = (
                margin_left
                + index / (len(history) - 1)
                * plot_width
            )

        y_position = (
            margin_top
            + (maximum_loss - loss)
            / loss_range
            * plot_height
        )

        return x_position, y_position

    train_points = []
    validation_points = []

    for index in range(len(history)):
        train_x, train_y = point_coordinates(
            index,
            train_losses[index]
        )

        validation_x, validation_y = point_coordinates(
            index,
            validation_losses[index]
        )

        train_points.append(
            f"{train_x:.2f},{train_y:.2f}"
        )

        validation_points.append(
            f"{validation_x:.2f},{validation_y:.2f}"
        )

    svg_lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" '
        f'width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}">',

        '<rect width="100%" height="100%" fill="white"/>',

        f'<text x="{width / 2}" y="25" '
        f'text-anchor="middle" '
        f'font-family="Arial" font-size="18" '
        f'font-weight="bold">'
        f'Task 1 Training and Validation Loss'
        f'</text>',

        f'<line x1="{margin_left}" '
        f'y1="{margin_top}" '
        f'x2="{margin_left}" '
        f'y2="{margin_top + plot_height}" '
        f'stroke="black"/>',

        f'<line x1="{margin_left}" '
        f'y1="{margin_top + plot_height}" '
        f'x2="{margin_left + plot_width}" '
        f'y2="{margin_top + plot_height}" '
        f'stroke="black"/>',

        f'<polyline points="{" ".join(train_points)}" '
        f'fill="none" stroke="#1f77b4" '
        f'stroke-width="3"/>',

        f'<polyline points="{" ".join(validation_points)}" '
        f'fill="none" stroke="#d62728" '
        f'stroke-width="3"/>',

        f'<text x="{margin_left + plot_width / 2}" '
        f'y="{height - 20}" '
        f'text-anchor="middle" '
        f'font-family="Arial" font-size="14">'
        f'Epoch'
        f'</text>',

        f'<text x="20" '
        f'y="{margin_top + plot_height / 2}" '
        f'text-anchor="middle" '
        f'transform="rotate(-90 20 '
        f'{margin_top + plot_height / 2})" '
        f'font-family="Arial" font-size="14">'
        f'Cross-Entropy Loss'
        f'</text>',

        f'<line x1="{width - 230}" y1="55" '
        f'x2="{width - 200}" y2="55" '
        f'stroke="#1f77b4" stroke-width="3"/>',

        f'<text x="{width - 190}" y="60" '
        f'font-family="Arial" font-size="13">'
        f'Training loss'
        f'</text>',

        f'<line x1="{width - 230}" y1="80" '
        f'x2="{width - 200}" y2="80" '
        f'stroke="#d62728" stroke-width="3"/>',

        f'<text x="{width - 190}" y="85" '
        f'font-family="Arial" font-size="13">'
        f'Validation loss'
        f'</text>',

        '</svg>'
    ]

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as file:
        file.write("\n".join(svg_lines))


loss_curve_path = "task1_loss_curves.svg"

create_loss_curve_svg(
    history=history,
    output_path=loss_curve_path
)

print("Saved:", metrics_report_path)
print("Saved: task1_model_configuration.json")
print("Saved:", loss_curve_path)

Saved: metrics_report.csv
Saved: task1_model_configuration.json
Saved: task1_loss_curves.svg


In [38]:
results_markdown = f"""
# DATA266 Task 1: GPT From Scratch

## Student and Reproducibility Information

- SID4: {SID4}
- Seed: {SEED}
- Slice: {SLICE}
- HP_ID: {HP_ID}
- CLS_A: {CLS_A}
- CLS_B: {CLS_B}
- Device: {DEVICE}
- GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available"}
- Python version: {platform.python_version()}
- PyTorch version: {torch.__version__}

## Dataset and Preprocessing

- Dataset: TinyStories
- Tokenization: Character-level
- Training sequences: {CONFIG["train_sequences"]:,}
- Validation sequences: {CONFIG["val_sequences"]:,}
- Sequence input length: {CONFIG["block_size"]}
- Vocabulary size: {VOCAB_SIZE}
- Training and validation split: Independently shuffled using seed {SEED}
- Character dictionaries: `char_to_idx` and `idx_to_char`

## Model Architecture

- Transformer blocks: {CONFIG["n_layers"]}
- Embedding dimension: {CONFIG["d_model"]}
- Attention heads: {CONFIG["n_heads"]}
- Dimension per attention head: {CONFIG["d_model"] // CONFIG["n_heads"]}
- Feed-forward dimension: {CONFIG["d_ff"]}
- Activation: {CONFIG["activation"]}
- Normalization: {CONFIG["normalization"]}
- Dropout: {CONFIG["dropout"]}
- Positional embeddings: Learnable
- Total parameters: {model_parameter_count:,}
- Attention implementation: Manual scaled dot-product attention
- Causal masking: Implemented and numerically verified
- Prebuilt Transformer or attention modules: Not used

## Training Configuration

- Optimizer: AdamW
- Learning rate: {CONFIG["learning_rate"]}
- Weight decay: {CONFIG["weight_decay"]}
- Warm-up steps: {CONFIG["warmup_steps"]}
- Scheduler: Linear warm-up followed by cosine decay
- Gradient clipping: {CONFIG["gradient_clip"]}
- Batch size: {CONFIG["batch_size"]}
- Epochs: {CONFIG["epochs"]}

## Final Evaluation Results

| Metric | Value |
|---|---:|
| Training cross-entropy loss | {final_metrics["training_cross_entropy_loss"]:.6f} |
| Validation cross-entropy loss | {final_metrics["validation_cross_entropy_loss"]:.6f} |
| Training perplexity | {final_metrics["training_perplexity"]:.6f} |
| Validation perplexity | {final_metrics["validation_perplexity"]:.6f} |
| Training bits per character | {final_metrics["training_bits_per_character"]:.6f} |
| Validation bits per character | {final_metrics["validation_bits_per_character"]:.6f} |
| Generalization gap | {final_metrics["generalization_gap"]:.6f} |
| Training top-1 accuracy | {final_metrics["training_top_1_accuracy"]:.6f} |
| Validation top-1 accuracy | {final_metrics["validation_top_1_accuracy"]:.6f} |
| Average gradient norm | {final_metrics["average_gradient_norm"]:.6f} |
| Maximum gradient norm | {final_metrics["maximum_gradient_norm"]:.6f} |
| Total NaN count | {final_metrics["total_nan_count"]} |
| Total loss spike count | {final_metrics["total_loss_spike_count"]} |
| Parameter count | {final_metrics["parameter_count"]:,} |
| Training tokens/second | {final_metrics["training_tokens_per_second"]:.2f} |
| Generation tokens/second | {final_metrics["generation_tokens_per_second"]:.2f} |
| Peak GPU memory GB | {final_metrics["peak_gpu_memory_gb"]:.4f} |
| Total training time seconds | {final_metrics["total_training_time_seconds"]:.2f} |

## Generation Diversity Results

| Metric | Average Value |
|---|---:|
| Distinct-1 | {final_metrics["distinct_1"]:.6f} |
| Distinct-2 | {final_metrics["distinct_2"]:.6f} |
| Distinct-3 | {final_metrics["distinct_3"]:.6f} |
| Repeated 4-gram rate | {final_metrics["repeated_4gram_rate"]:.6f} |

Generation used temperature sampling with temperature {CONFIG["temperature"]}.

## Observations

The model successfully learned common TinyStories character and word patterns.
The validation loss decreased throughout training, and the final validation
perplexity was approximately {final_metrics["validation_perplexity"]:.2f}.
The model generated recognizable story structures, but some samples contained
grammar errors, repeated phrases, abrupt topic changes, and semantically
inconsistent descriptions.

The complete failure analysis is stored in `task1_failure_analysis.md`.

## Individual Task 1 Artifacts

- `metrics_report.csv`
- `task1_model_configuration.json`
- `task1_training_history.json`
- `task1_generated_samples.txt`
- `task1_loss_curves.svg`
- `task1_failure_analysis.md`
- `task1_checkpoints/task1_best_model.pt`
- `task1_checkpoints/task1_epoch_01.pt` through `task1_epoch_10.pt`
"""

with open(
    "results.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(results_markdown)


manifest = {
    "assignment": "DATA266 Lab Task 1",
    "student_identifier": "Yashashree",
    "SID4": SID4,
    "seed": SEED,
    "device": str(DEVICE),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "Not available"
    ),
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "dataset": "roneneldan/TinyStories",
    "checkpoint": "task1_checkpoints/task1_best_model.pt",
    "results_file": "results.md",
    "metrics_file": "metrics_report.csv",
    "failure_analysis_file": "task1_failure_analysis.md",
    "loss_curve_file": "task1_loss_curves.svg"
}

with open(
    "task1_manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        manifest,
        file,
        indent=2
    )

print("Saved results.md")
print("Saved task1_manifest.json")

Saved results.md
Saved task1_manifest.json
